# Smart Beta and Factor Investing

FINM 36700 HW6: Smart-Beta

Group: Avi Rastogi, Lucas Albin Lazcano, Mengting Wang, Siena Verprauskus

## Setup

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

FILE = "factor_pricing_data.xlsx"
factors = pd.read_excel(FILE, sheet_name="factors (excess returns)", index_col="Date", parse_dates=True)
ports   = pd.read_excel(FILE, sheet_name="portfolios (excess returns)", index_col="Date", parse_dates=True)
ports.columns = ports.columns.str.strip()

print(f"{factors.index[0]:%Y-%m} to {factors.index[-1]:%Y-%m}, {len(factors)} months, {ports.shape[1]} industries")

1980-01 to 2025-04, 544 months, 49 industries


---
# 2. The Factors

Data: `data/factor_pricing_data.xlsx` — monthly excess returns for MKT and five additional factors (all already excess returns).

## 2.1
Analyze the factors (as with the Fama-French factors in Homework 4). Compare univariate statistics:
- mean
- volatility
- Sharpe

In [2]:
def summary(df):
    return pd.DataFrame({"Mean": df.mean() * 12,
                         "Vol": df.std() * np.sqrt(12),
                         "Sharpe": df.mean() / df.std() * np.sqrt(12),
                         "t-stat": df.mean() / df.std() * np.sqrt(len(df))})

summary(factors).round(3)

,Mean,Vol,Sharpe,t-stat
MKT,0.085,0.156,0.543,3.657
SMB,0.005,0.101,0.053,0.360
HML,0.027,0.109,0.243,1.639
RMW,0.045,0.083,0.543,3.659
CMA,0.028,0.072,0.382,2.571
UMD,0.062,0.154,0.402,2.709


**Answer:**

All six factors have positive average returns over 1980–2025, but they're very different. MKT and RMW have the best Sharpe ratios (both 0.54), and RMW gets there with about half the market's volatility (8.3% vs 15.6%). UMD (0.40) and CMA (0.38) are in the middle. HML (0.24) and especially SMB (0.05) are weak, and SMB's mean is only about 0.5% a year.

## 2.2
Based on the factor statistics above:
- Does each factor have a positive risk premium (positive expected excess return)?
- How have the factors performed since the time of the case (2015-present)?

In [3]:
pd.concat({"1980–2014": summary(factors.loc[:"2014"]), "2015–present": summary(factors.loc["2015":])}, axis=1).round(3)

1980–2014                      2015–present                     
         Mean    Vol Sharpe t-stat         Mean    Vol Sharpe t-stat
MKT     0.078  0.156  0.502  2.972        0.107  0.159  0.677  2.178
SMB     0.015  0.101  0.151  0.895       -0.028  0.104 -0.269 -0.864
HML     0.039  0.101  0.384  2.271       -0.016  0.131 -0.119 -0.384
RMW     0.045  0.086  0.527  3.118        0.045  0.073  0.613  1.970
CMA     0.040  0.069  0.575  3.400       -0.013  0.082 -0.158 -0.507
UMD     0.073  0.158  0.460  2.721        0.025  0.138  0.182  0.586

**Answer:**

Over the full sample all six are positive, but only MKT, RMW, CMA and UMD are clearly different from zero (t-stats of 2.6 to 3.7). SMB (t = 0.4) and HML (t = 1.6) aren't.

Since 2015 they've split into two groups. MKT (Sharpe 0.68) and RMW (0.61) kept working, while UMD weakened a lot (2.5% a year, down from 7.3%). SMB, HML and CMA all had negative average returns. With only about ten years of data, none of those four post-2015 means is significantly different from zero, though. So we can't say those premia are gone, just that the last decade was bad for them.

## 2.3
Report the correlation matrix across the six factors.
- Does the construction method succeed in keeping correlations small?
- Fama and French say HML is somewhat redundant in their 5-factor model. Does this seem to be the case?

In [4]:
factors.corr().round(2)

,MKT,SMB,HML,RMW,CMA,UMD
MKT,1.00,0.23,-0.21,-0.25,-0.35,-0.18
SMB,0.23,1.00,-0.03,-0.41,-0.05,-0.06
HML,-0.21,-0.03,1.00,0.22,0.68,-0.21
RMW,-0.25,-0.41,0.22,1.00,0.14,0.07
CMA,-0.35,-0.05,0.68,0.14,1.00,0.00
UMD,-0.18,-0.06,-0.21,0.07,0.00,1.00


In [5]:
# is HML redundant? regress it on the other four FF5 factors
fit = sm.OLS(factors["HML"], sm.add_constant(factors[["MKT", "SMB", "RMW", "CMA"]])).fit()
print(f"alpha = {fit.params['const'] * 12:.2%} a year (t = {fit.tvalues['const']:.2f}), R² = {fit.rsquared:.2f}")
pd.DataFrame({"coef": fit.params, "t-stat": fit.tvalues}).round(2)

alpha = -1.57% a year (t = -1.29), R² = 0.49


,coef,t-stat
const,-0.00,-1.29
MKT,0.04,1.78
SMB,0.07,1.88
RMW,0.22,4.82
CMA,1.02,20.63


**Answer:**

Mostly yes. Most correlations are 0.35 or less in absolute value, and some are close to zero (UMD–CMA 0.00, SMB–HML −0.03). The two big ones make sense. HML–CMA is 0.68 because value firms tend to be low-investment firms, and SMB–RMW is −0.41 because small firms tend to be less profitable.

HML does look redundant. Regressing it on MKT, SMB, RMW and CMA gives an R² of 0.49, a CMA loading of about 1.0 (t = 21), and an alpha of −1.6% a year that isn't significant (t = −1.3). So the other four factors already reproduce HML's average return.

## 2.4
Report the tangency weights for a portfolio of these 6 factors.
- Which factors seem most important? And least?
- Are the factors with low mean returns still useful?
- Re-do the tangency portfolio with only MKT, SMB, HML, UMD. Which factors get high/low tangency weights now?

What do you conclude about the importance or unimportance of these styles?

In [6]:
def tangency(df):
    w = np.linalg.solve(df.cov(), df.mean())
    w = w / w.sum()
    r = df @ w
    return pd.Series(list(w) + [r.mean() / r.std() * np.sqrt(12)], index=list(df.columns) + ["Sharpe"])

pd.DataFrame({"6 factors": tangency(factors),
              "4 factors": tangency(factors[["MKT", "SMB", "HML", "UMD"]])}).round(3)

,6 factors,4 factors
CMA,0.317,NaN
HML,-0.020,0.368
MKT,0.216,0.371
RMW,0.306,NaN
SMB,0.067,-0.054
Sharpe,1.194,0.906
UMD,0.114,0.314


**Answer:**

With all six, the weight goes mostly to CMA (32%), RMW (31%) and MKT (22%). UMD gets 11%, SMB 7%, and HML is slightly negative (−2%). So CMA, RMW and MKT look most important and HML least, which matches 2.3.

Low-mean factors can still be useful. CMA has only a 2.8% mean but gets the biggest weight, because its volatility is low (7.3%) and it moves against the market (−0.35). The optimizer cares about what a factor adds to the portfolio, not its own Sharpe ratio.

With only MKT, SMB, HML and UMD, the weights shift to MKT (37%), HML (37%) and UMD (31%), with SMB slightly negative (−5%). HML goes from −2% to 37% because, without CMA, it's the closest substitute. The tangency Sharpe drops from 1.19 to 0.91.

So a style's importance depends on what else is in the set. MKT matters in both sets and SMB in neither. Value matters only when investment (CMA) isn't there, and profitability and momentum add something the original FF3 factors don't.

---
# 3. Testing Modern LPMs

Models:
- **CAPM**: MKT
- **FF3**: MKT, SMB, HML
- **FF5**: MKT, SMB, HML, RMW, CMA
- **AQR**: MKT, HML, RMW, UMD

Time-series test (AQR example):

$$\tilde{r}^i_t = \alpha^i + \beta^{i,\mathrm{MKT}} \tilde{f}^{\mathrm{MKT}}_t + \beta^{i,\mathrm{HML}} \tilde{f}^{\mathrm{HML}}_t + \beta^{i,\mathrm{RMW}} \tilde{f}^{\mathrm{RMW}}_t + \beta^{i,\mathrm{UMD}} \tilde{f}^{\mathrm{UMD}}_t + \varepsilon_t$$

Data: monthly excess returns on $n=49$ industry portfolios. Risk-free rate not needed.

## 3.1
Test the AQR 4-factor model using time-series regressions. For each regression, report the estimated $\alpha$ and $R^2$.

In [7]:
models = {"CAPM": ["MKT"], "FF3": ["MKT", "SMB", "HML"],
          "FF5": ["MKT", "SMB", "HML", "RMW", "CMA"], "AQR": ["MKT", "HML", "RMW", "UMD"]}

def ts_test(cols):
    X = sm.add_constant(factors[cols])
    rows = {}
    for ind in ports:
        fit = sm.OLS(ports[ind], X).fit()
        rows[ind] = {"alpha": fit.params["const"] * 12, "t(alpha)": fit.tvalues["const"], "R²": fit.rsquared,
                     **{f"beta {c}": fit.params[c] for c in cols}}
    return pd.DataFrame(rows).T

ts = {m: ts_test(cols) for m, cols in models.items()}
ts["AQR"][["alpha", "t(alpha)", "R²"]].round(3)

,alpha,t(alpha),R²
Agric,0.011,0.400,0.340
Food,0.004,0.244,0.461
Soda,0.018,0.637,0.304
Beer,0.011,0.550,0.421
Smoke,0.043,1.436,0.266
Toys,-0.035,-1.283,0.509
Fun,0.041,1.592,0.610
Books,-0.035,-1.966,0.690
Hshld,-0.010,-0.643,0.559
Clths,-0.023,-1.049,0.618


**Answer:**

The table has the alpha and R² for each of the 49 industries. Alphas run from −5.4% a year (RlEst) to +6.9% (Softw), and R² goes from 0.05 (Gold) to 0.85 (BusSv).

## 3.2
Calculate the mean absolute error of the estimated alphas:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^n|\tilde{\alpha}^i|$$

- If the pricing model worked, should these alpha estimates be large or small? Why?
- Based on your MAE, does this support the pricing model or not?

In [8]:
aqr = ts["AQR"]
print(f"AQR MAE = {aqr['alpha'].abs().mean():.2%} a year; significant alphas: {(aqr['t(alpha)'].abs() > 1.96).sum()} of {len(aqr)}")
print(f"average industry excess return = {ports.mean().mean() * 12:.2%}; market premium = {factors['MKT'].mean() * 12:.2%}")

AQR MAE = 2.46% a year; significant alphas: 7 of 49
average industry excess return = 8.91%; market premium = 8.49%


**Answer:**

If the model worked, the alphas should be close to zero, because each industry's average return would be fully explained by its betas times the factor premia. Any alpha is a pricing error.

The AQR model's MAE is 2.5% a year, which is big compared with the 8.5% market premium and the 8.9% average industry return. 7 of the 49 alphas are significant at 5%, when you'd expect about 2 or 3 by chance. So this doesn't support the model very well.

## 3.3
Test the CAPM, FF3 and FF5.
- Report the MAE for each and compare with the AQR MAE.
- Which model fits best?

In [9]:
mae = pd.DataFrame({m: {"MAE": r["alpha"].abs().mean(), "# significant alphas": int((r["t(alpha)"].abs() > 1.96).sum())}
                    for m, r in ts.items()}).T
mae.style.format({"MAE": "{:.2%}", "# significant alphas": "{:.0f}"})

,MAE,# significant alphas
CAPM,2.14%,6
FF3,2.45%,7
FF5,3.15%,16
AQR,2.46%,7


**Answer:**

The CAPM fits best: an MAE of 2.14% with 6 significant alphas. FF3 (2.45%, 7) and AQR (2.46%, 7) are about the same, and FF5 is clearly the worst (3.15%, 16). More factors don't automatically price industries better. These factors were built to price portfolios sorted on size, value, profitability and investment, and industries load on them in ways their average returns don't reflect.

## 3.4
Does any particular factor seem especially important or unimportant for pricing? Should Fama and French use the Momentum factor?

In [10]:
# does momentum help? add UMD to FF5 and compare
ff5_umd = ts_test(models["FF5"] + ["UMD"])
print(f"FF5:       MAE {ts['FF5']['alpha'].abs().mean():.2%}, {(ts['FF5']['t(alpha)'].abs() > 1.96).sum()} significant alphas")
print(f"FF5 + UMD: MAE {ff5_umd['alpha'].abs().mean():.2%}, {(ff5_umd['t(alpha)'].abs() > 1.96).sum()} significant alphas")

FF5:       MAE 3.15%, 16 significant alphas
FF5 + UMD: MAE 2.87%, 14 significant alphas


**Answer:**

MKT does most of the work, since the CAPM alone has the lowest MAE. RMW and CMA don't help with pricing industries. Adding them (FF3 → FF5) raises the MAE from 2.45% to 3.15% and the significant alphas from 7 to 16, even though they got the biggest tangency weights in 2.4. Being useful for investing isn't the same as being useful for pricing.

On momentum, adding UMD to FF5 lowers the MAE from 3.15% to 2.87% and the significant alphas from 16 to 14. So it carries some pricing information FF5 is missing, and it would be reasonable for Fama and French to include it, although the gain on industry portfolios is small.

## 3.5
Report the average (across $n$) time-series $R^2$ for each of the four models.
- Do these models lead to high time-series $R^2$? Would these factors be good for a Linear Factor Decomposition of the assets?

In [11]:
pd.DataFrame({m: {"avg R²": r["R²"].mean(), "min R²": r["R²"].min(), "lowest": r["R²"].idxmin()}
              for m, r in ts.items()}).T

,avg R²,min R²,lowest
CAPM,0.524438,0.048201,Gold
FF3,0.569632,0.05412,Gold
FF5,0.593895,0.067912,Gold
AQR,0.573537,0.048796,Gold


**Answer:**

The average R² is 0.52 for the CAPM, 0.57 for FF3, 0.59 for FF5 and 0.57 for AQR. So the market does most of the explaining, and the extra factors add only a few points.

That's not high enough for a good linear factor decomposition. Roughly half of each industry's variance is left unexplained, and some industries are barely explained at all (Gold at about 0.05). R² and pricing are also different questions: FF5 has the highest R² but the worst alphas.

## 3.6
Re-test all four models using cross-sectional regressions **with an intercept**.
- Report the time-series factor premia (sample averages) and compare to the cross-sectionally estimated premia. Do they differ substantially?
- Report the MAE of the cross-sectional residuals for each model. How do they compare to the MAE of the time-series alphas?

In [12]:
mean_rets = ports.mean() * 12

cs = {}
for m, cols in models.items():
    betas = ts[m][[f"beta {c}" for c in cols]]
    betas.columns = cols
    cs[m] = sm.OLS(mean_rets, sm.add_constant(betas)).fit()

premia = pd.DataFrame({m: fit.params for m, fit in cs.items()}).T
premia.loc["Time-series (sample mean)"] = factors.mean() * 12
premia[["const", "MKT", "SMB", "HML", "RMW", "CMA", "UMD"]].style.format("{:.2%}", na_rep="—")

,const,MKT,SMB,HML,RMW,CMA,UMD
CAPM,8.69%,0.21%,—,—,—,—,—
FF3,6.24%,3.76%,-4.47%,-1.99%,—,—,—
FF5,6.00%,3.84%,-4.70%,-2.58%,2.31%,-2.43%,—
AQR,7.58%,1.44%,—,-3.26%,2.13%,—,0.19%
Time-series (sample mean),—,8.49%,0.54%,2.65%,4.51%,2.77%,6.18%


In [13]:
pd.DataFrame({m: {"TS MAE (alpha)": ts[m]["alpha"].abs().mean(),
                  "CS MAE (residual)": cs[m].resid.abs().mean(),
                  "CS R²": cs[m].rsquared} for m in models}).T.style.format({"TS MAE (alpha)": "{:.2%}", "CS MAE (residual)": "{:.2%}", "CS R²": "{:.2f}"})

,TS MAE (alpha),CS MAE (residual),CS R²
CAPM,2.14%,1.55%,0.00
FF3,2.45%,1.20%,0.38
FF5,3.15%,1.19%,0.42
AQR,2.46%,1.38%,0.21


**Answer:**

They're very different. The cross-sectional market premium is only 0.2% (CAPM) to 3.8% (FF3, FF5), against 8.5% in the time series. SMB, HML and CMA get negative premia even though their sample means are positive. RMW gets about 2.2% vs 4.5%, and UMD about 0.2% vs 6.2%. The intercepts are 6–9% a year, when they should be zero.

The cross-sectional MAEs (1.2%–1.6%) are lower than the time-series ones (2.1%–3.2%), but that's expected. The cross-sectional regression picks its premia and intercept to fit these exact 49 average returns, while the time-series test fixes the premia at the factor means. The ranking also flips: FF5 fits best cross-sectionally (R² of 0.42) and the CAPM worst (R² of about 0). Since the premia it needs don't match the factors' actual returns, that isn't evidence the models work.